# 🏆 Loyalty Program Design & Point-Level Segmentation

> **Dataset:** Online Retail (UCI Machine Learning Repository)  
> **Source:** https://archive.ics.uci.edu/dataset/352/online+retail  
> **Goal:** Design a points-based loyalty programme for an e-commerce retailer. Assign customers to tiers based on purchase behaviour, calculate earned points, and recommend rewards with increasing perceived value at each level.

---

## Business Question

> *"How do we reward our best customers in a way that feels meaningful at every level — and incentivises everyone to spend more?"*

### Programme Architecture

| Tier | Name | Points Threshold | Earn Rate | Goal |
|---|---|---|---|---|
| 1 | Bronze | 0 – 499 pts | 1 pt / £1 | Activate new customers |
| 2 | Silver | 500 – 1,499 pts | 1.25 pts / £1 | Grow mid-tier spend |
| 3 | Gold | 1,500 – 3,999 pts | 1.5 pts / £1 | Retain high-value customers |
| 4 | Platinum | 4,000+ pts | 2 pts / £1 | Lock in top customers |

**Points = Total Spend × Earn Rate**  
**Perceived value of rewards increases with tier** — from discounts to exclusive experiences.

---
## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── Palette ─────────────────────────────────────────────────
CHOCOLATE  = '#3d2314'
BROWN      = '#7a4f35'
CAMEL      = '#b89a74'
TERRACOTTA = '#c4694f'
SAND       = '#d9cdb8'
PARCHMENT  = '#ede5d4'

# Tier colours
TIER_COLORS = {
    'Bronze':   '#cd7f32',
    'Silver':   '#a8a9ad',
    'Gold':     '#d4af37',
    'Platinum': '#4a4e69',
}

plt.rcParams.update({
    'figure.facecolor':  PARCHMENT,
    'axes.facecolor':    '#f5f0e8',
    'axes.edgecolor':    SAND,
    'axes.labelcolor':   BROWN,
    'axes.titlecolor':   CHOCOLATE,
    'axes.titlesize':    13,
    'axes.titleweight':  'normal',
    'axes.labelsize':    10,
    'xtick.color':       BROWN,
    'ytick.color':       BROWN,
    'grid.color':        SAND,
    'grid.linestyle':    '--',
    'grid.alpha':        0.5,
    'font.family':       'serif',
    'text.color':        CHOCOLATE,
})

import os
os.makedirs('outputs', exist_ok=True)
print('Libraries loaded ✅')

---
## 2. Load & Clean Data

In [ ]:
df = pd.read_excel('data/online_retail.xlsx', dtype={'CustomerID': str})

# Clean
df = df[~df['InvoiceNo'].astype(str).str.startswith('C')]
df = df.dropna(subset=['CustomerID'])
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)]
df['Revenue'] = df['Quantity'] * df['UnitPrice']
df_uk = df[df['Country'] == 'United Kingdom'].copy()

print(f'UK customers: {df_uk.CustomerID.nunique():,}')
print(f'Date range:   {df_uk.InvoiceDate.min().date()} → {df_uk.InvoiceDate.max().date()}')
df_uk.head()

---
## 3. Build Customer Summary

Aggregate each customer's total spend, purchase frequency, average order value, and recency.

In [ ]:
snapshot_date = df_uk['InvoiceDate'].max() + pd.Timedelta(days=1)

customers = df_uk.groupby('CustomerID').agg(
    TotalSpend      = ('Revenue', 'sum'),
    NumOrders       = ('InvoiceNo', 'nunique'),
    NumItems        = ('Quantity', 'sum'),
    AvgOrderValue   = ('Revenue', lambda x: x.groupby(df_uk.loc[x.index, 'InvoiceNo']).sum().mean()),
    FirstPurchase   = ('InvoiceDate', 'min'),
    LastPurchase    = ('InvoiceDate', 'max'),
).reset_index()

customers['DaysSinceLastPurchase'] = (snapshot_date - customers['LastPurchase']).dt.days
customers['TenureDays'] = (customers['LastPurchase'] - customers['FirstPurchase']).dt.days

print(f'Customers: {len(customers):,}')
print(f'Avg spend: £{customers.TotalSpend.mean():.2f}')
customers.describe().round(2)

---
## 4. Assign Loyalty Points & Tiers

Points are calculated based on total spend. The earn rate increases with tier — rewarding customers who have already reached higher levels with faster point accumulation.

In [ ]:
# ── Step 1: initial points at base rate (1 pt / £1) ─────────
customers['BasePoints'] = customers['TotalSpend'].round(0).astype(int)

# ── Step 2: assign tier based on base points ─────────────────
def assign_tier(points):
    if points >= 4000:  return 'Platinum'
    elif points >= 1500: return 'Gold'
    elif points >= 500:  return 'Silver'
    else:                return 'Bronze'

customers['Tier'] = customers['BasePoints'].apply(assign_tier)

# ── Step 3: apply tier earn rate ─────────────────────────────
earn_rates = {'Bronze': 1.0, 'Silver': 1.25, 'Gold': 1.5, 'Platinum': 2.0}
customers['EarnRate'] = customers['Tier'].map(earn_rates)
customers['FinalPoints'] = (customers['TotalSpend'] * customers['EarnRate']).round(0).astype(int)

# ── Step 4: points to next tier ──────────────────────────────
tier_thresholds = {'Bronze': 500, 'Silver': 1500, 'Gold': 4000, 'Platinum': None}

def points_to_next(row):
    next_thresh = tier_thresholds[row['Tier']]
    if next_thresh is None: return 0
    return max(0, next_thresh - row['BasePoints'])

customers['PointsToNextTier'] = customers.apply(points_to_next, axis=1)

# ── Summary ───────────────────────────────────────────────────
tier_summary = customers.groupby('Tier').agg(
    Customers       = ('CustomerID', 'count'),
    AvgSpend        = ('TotalSpend', 'mean'),
    AvgPoints       = ('FinalPoints', 'mean'),
    TotalRevenue    = ('TotalSpend', 'sum'),
    AvgOrders       = ('NumOrders', 'mean'),
    AvgAOV          = ('AvgOrderValue', 'mean'),
).round(2)

tier_order = ['Bronze', 'Silver', 'Gold', 'Platinum']
tier_summary = tier_summary.reindex(tier_order)
tier_summary['RevenueShare'] = (tier_summary['TotalRevenue'] / tier_summary['TotalRevenue'].sum()).map('{:.1%}'.format)
tier_summary

---
## 5. Reward Design by Tier

Rewards are designed so that **perceived value increases meaningfully at each tier**. The goal is to make the next tier always feel worth reaching.

In [ ]:
rewards = {
    'Bronze': {
        'Earn Rate':         '1 pt per £1 spent',
        'Welcome Reward':    '£5 voucher on first redemption',
        'Birthday Reward':   '10% off on birthday month',
        'Redemption':        '100 pts = £1 off',
        'Exclusive Access':  'Early sale access (24h)',
        'Perceived Value':   '★☆☆☆',
        'Typical Value/yr':  '~£8–15',
    },
    'Silver': {
        'Earn Rate':         '1.25 pts per £1 spent',
        'Welcome Reward':    '£10 voucher on tier upgrade',
        'Birthday Reward':   '15% off + free gift wrapping',
        'Redemption':        '100 pts = £1.25 off',
        'Exclusive Access':  'Early sale access (48h) + members newsletter',
        'Perceived Value':   '★★☆☆',
        'Typical Value/yr':  '~£25–45',
    },
    'Gold': {
        'Earn Rate':         '1.5 pts per £1 spent',
        'Welcome Reward':    '£20 voucher + free shipping for 3 months',
        'Birthday Reward':   '20% off + surprise gift',
        'Redemption':        '100 pts = £1.50 off',
        'Exclusive Access':  'Private sale + dedicated support line',
        'Perceived Value':   '★★★☆',
        'Typical Value/yr':  '~£80–150',
    },
    'Platinum': {
        'Earn Rate':         '2 pts per £1 spent',
        'Welcome Reward':    '£50 voucher + free annual shipping',
        'Birthday Reward':   '25% off + curated gift box',
        'Redemption':        '100 pts = £2 off',
        'Exclusive Access':  'VIP events + personal shopper + first access to new products',
        'Perceived Value':   '★★★★',
        'Typical Value/yr':  '~£250–500+',
    },
}

rewards_df = pd.DataFrame(rewards).T
rewards_df.index.name = 'Tier'
print('Reward structure by tier:')
rewards_df

---
## 6. Visualisations

### 6a. Tier Distribution — Customers & Revenue

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors = [TIER_COLORS[t] for t in tier_order]

# Customer count
ax = axes[0]
vals = tier_summary['Customers'].values
bars = ax.bar(tier_order, vals, color=colors, edgecolor='white', linewidth=0.5)
for bar, v in zip(bars, vals):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+10,
            f'{v:,}', ha='center', fontsize=9, color=CHOCOLATE)
ax.set_title('Customers per Tier')
ax.set_ylabel('Customers')
ax.grid(axis='y')

# Revenue share donut
ax2 = axes[1]
rev_vals = tier_summary['TotalRevenue'].values
wedges, texts, autotexts = ax2.pie(
    rev_vals, labels=tier_order, autopct='%1.1f%%',
    colors=colors, startangle=90,
    wedgeprops=dict(width=0.55, edgecolor='white', linewidth=2),
    pctdistance=0.75
)
for at in autotexts:
    at.set_fontsize(8)
ax2.set_title('Revenue Share by Tier')

# Avg spend per tier
ax3 = axes[2]
avg_vals = tier_summary['AvgSpend'].values
bars3 = ax3.bar(tier_order, avg_vals, color=colors, edgecolor='white', linewidth=0.5)
for bar, v in zip(bars3, avg_vals):
    ax3.text(bar.get_x()+bar.get_width()/2, bar.get_height()+5,
             f'£{v:,.0f}', ha='center', fontsize=9, color=CHOCOLATE)
ax3.set_title('Average Spend per Tier')
ax3.set_ylabel('Avg Total Spend (£)')
ax3.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'£{x:,.0f}'))
ax3.grid(axis='y')

plt.suptitle('Loyalty Programme — Tier Overview', fontsize=14, color=CHOCOLATE)
plt.tight_layout()
plt.savefig('outputs/loyalty_tier_overview.png', dpi=150, bbox_inches='tight')
plt.show()

### 6b. Points Distribution per Tier

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

for tier in tier_order:
    subset = customers[customers['Tier'] == tier]['FinalPoints']
    ax.hist(subset, bins=40, alpha=0.7, label=tier,
            color=TIER_COLORS[tier], edgecolor='white', linewidth=0.3)

# Threshold lines
for thresh, label, color in zip(
    [500, 1500, 4000],
    ['Silver threshold\n500 pts', 'Gold threshold\n1,500 pts', 'Platinum threshold\n4,000 pts'],
    [TIER_COLORS['Silver'], TIER_COLORS['Gold'], TIER_COLORS['Platinum']]
):
    ax.axvline(thresh, color=color, linestyle='--', linewidth=1.5, alpha=0.8)
    ax.text(thresh+30, ax.get_ylim()[1]*0.85, label, fontsize=7.5,
            color=color, style='italic')

ax.set_title('Points Distribution by Tier')
ax.set_xlabel('Loyalty Points Earned')
ax.set_ylabel('Number of Customers')
ax.legend(title='Tier', framealpha=0.7)
ax.grid(axis='y')
ax.set_xlim(0, customers['FinalPoints'].quantile(0.97))

plt.tight_layout()
plt.savefig('outputs/loyalty_points_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

### 6c. Upgrade Opportunity Map — Who is Close to the Next Tier?

In [ ]:
# Customers NOT in Platinum who are within 20% of next tier threshold
upgrade_candidates = customers[
    (customers['Tier'] != 'Platinum') &
    (customers['PointsToNextTier'] > 0) &
    (customers['PointsToNextTier'] / customers['BasePoints'] < 0.2)
].copy()

upgrade_candidates['PctToNext'] = (
    customers['BasePoints'] /
    customers['Tier'].map({'Bronze': 500, 'Silver': 1500, 'Gold': 4000, 'Platinum': 99999})
).clip(0, 1) * 100

print(f'Customers within 20% of next tier: {len(upgrade_candidates):,}')
print(f'\nBy current tier:')
print(upgrade_candidates.groupby('Tier')['CustomerID'].count().reindex(['Bronze','Silver','Gold']))

fig, ax = plt.subplots(figsize=(11, 5))

for tier in ['Bronze', 'Silver', 'Gold']:
    subset = upgrade_candidates[upgrade_candidates['Tier'] == tier]
    ax.scatter(
        subset['TotalSpend'], subset['PointsToNextTier'],
        alpha=0.5, s=30, color=TIER_COLORS[tier], label=f'{tier} → next tier',
        edgecolors='none'
    )

ax.set_title('Upgrade Opportunity: Spend vs Points Needed to Next Tier')
ax.set_xlabel('Total Spend to Date (£)')
ax.set_ylabel('Points Still Needed for Next Tier')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'£{x:,.0f}'))
ax.legend(framealpha=0.7)
ax.grid(True, alpha=0.4)

plt.tight_layout()
plt.savefig('outputs/loyalty_upgrade_map.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'\n💡 These {len(upgrade_candidates):,} customers are prime targets for a "You\'re almost there!" nudge campaign.')

### 6d. Reward Cost vs Revenue per Tier

In [ ]:
# Estimate reward cost: assume 5% of points are redeemed at £1/100pts
REDEMPTION_RATE = 0.05
customers['EstRewardCost'] = customers['FinalPoints'] * REDEMPTION_RATE * 0.01

cost_revenue = customers.groupby('Tier').agg(
    Revenue    = ('TotalSpend', 'sum'),
    RewardCost = ('EstRewardCost', 'sum')
).reindex(tier_order)
cost_revenue['CostRatio'] = cost_revenue['RewardCost'] / cost_revenue['Revenue'] * 100
cost_revenue['ROI'] = (cost_revenue['Revenue'] - cost_revenue['RewardCost']) / cost_revenue['RewardCost']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Revenue vs Cost
x = np.arange(len(tier_order))
w = 0.35
ax = axes[0]
ax.bar(x - w/2, cost_revenue['Revenue'], w, label='Revenue',
       color=[TIER_COLORS[t] for t in tier_order], alpha=0.85, edgecolor='white')
ax.bar(x + w/2, cost_revenue['RewardCost'], w, label='Est. Reward Cost',
       color=TERRACOTTA, alpha=0.7, edgecolor='white')
ax.set_xticks(x)
ax.set_xticklabels(tier_order)
ax.set_title('Revenue vs Estimated Reward Cost by Tier')
ax.set_ylabel('£')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'£{x:,.0f}'))
ax.legend(framealpha=0.7)
ax.grid(axis='y')

# Cost ratio
ax2 = axes[1]
bars = ax2.bar(tier_order, cost_revenue['CostRatio'],
               color=[TIER_COLORS[t] for t in tier_order], edgecolor='white')
for bar, v in zip(bars, cost_revenue['CostRatio']):
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.05,
             f'{v:.2f}%', ha='center', fontsize=9, color=CHOCOLATE)
ax2.set_title('Reward Cost as % of Revenue by Tier')
ax2.set_ylabel('Cost Ratio (%)')
ax2.grid(axis='y')

plt.suptitle('Programme Economics', fontsize=14, color=CHOCOLATE)
plt.tight_layout()
plt.savefig('outputs/loyalty_programme_economics.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7. Export

In [ ]:
customers[[
    'CustomerID','TotalSpend','NumOrders','AvgOrderValue',
    'DaysSinceLastPurchase','BasePoints','EarnRate','FinalPoints',
    'Tier','PointsToNextTier'
]].sort_values('FinalPoints', ascending=False).to_csv('outputs/loyalty_scores.csv', index=False)

rewards_df.to_csv('outputs/reward_catalogue.csv')

print('Exported:')
print('  → outputs/loyalty_scores.csv ✅')
print('  → outputs/reward_catalogue.csv ✅')

---
## 8. Key Findings & Recommendations

---

### 🔍 Findings

| # | Finding |
|---|---|
| 1 | **Platinum customers are few but drive a disproportionate share of revenue** — classic power-law, consistent with LTV analysis |
| 2 | **Bronze tier is the largest by volume** — most customers have low engagement; activation campaigns are needed |
| 3 | **A significant cluster of customers sits just below tier thresholds** — prime targets for nudge campaigns |
| 4 | **Reward cost as % of revenue is sustainable** — estimated below 1% across all tiers at a 5% redemption rate |
| 5 | **Gold and Platinum customers have significantly higher AOV** — rewards should emphasise exclusivity over discounts at these tiers |

---

### 💡 Recommendations

**1. Launch a "You're almost there" campaign**  
Target customers within 20% of the next tier threshold with a personalised email showing exactly how many points they need and what rewards await them. This is the highest-ROI loyalty campaign available.

**2. Differentiate rewards by perceived value, not just discount depth**  
Bronze customers respond to immediate savings (vouchers, % off). Platinum customers value exclusivity and experience (personal shopper, VIP events). Align reward design accordingly.

**3. Introduce a double-points event to accelerate Bronze → Silver upgrades**  
A seasonal double-points weekend can move large numbers of Bronze customers into Silver, improving long-term retention without significant cost.

**4. Protect Platinum with non-monetary perks**  
Top customers should feel recognised beyond points. Dedicated support, early product access, and handwritten thank-you notes have high perceived value at near-zero cost.

**5. Monitor redemption rate closely**  
If redemption rises above 10%, the reward cost % of revenue will increase significantly. Set quarterly reviews of programme economics.

---

*Analysis by Danai Avratoglou | Dataset: UCI Online Retail | Tools: Python, pandas, matplotlib*